# Phase 2 — CNN Baseline on Kaggle
**Yale Brain Mets Longitudinal** · 3D CNN + Grad-CAM  
Pseudo-labels: 10% increase in POST signal → progressive (1), else stable (0)

| Dataset | Kaggle ID |
|---------|-----------|
| Processed NIfTI | `mohamedmohamed23/yale-processed-nifti` |
| Manifest CSV | `mohamedmohamed23/yale-processed-manifest` |

Run on **P100** (recommended) or **T4** — auto-detected.

In [ ]:
"""
CELL 1 — Environment check & install missing packages
Run this ONCE. On Kaggle the packages are pre-installed.
"""
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import nibabel
except ImportError:
    pip_install("nibabel")

try:
    from tqdm.auto import tqdm
except ImportError:
    pip_install("tqdm")

try:
    from scipy.ndimage import zoom
except ImportError:
    pip_install("scipy")

print("All packages ready.")


In [ ]:
"""
CELL 2 — All imports & config
"""
from pathlib import Path
import json, warnings, time, os
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.ndimage import zoom
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, ConfusionMatrixDisplay)
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore")

# ── Kaggle auto-detection ────────────────────────────────────────────────────
ON_KAGGLE = os.path.exists("/kaggle/working")

if ON_KAGGLE:
    BASE_DIR         = Path("/kaggle/working")
    MANIFEST         = Path("/kaggle/input/datasets/mohamedmohamed23/yale-processed-manifest/processed_manifest.csv")
    CKPT_DIR         = Path("/kaggle/working/checkpoints")
    LOCAL_PROCESSED  = None
    KAGGLE_PROCESSED = "/kaggle/input/yale-processed-nifti"
else:
    BASE_DIR         = Path("/home/moamed/canada_me/explainable_diseas/implementation")
    MANIFEST         = BASE_DIR / "outputs" / "processed_manifest.csv"
    CKPT_DIR         = BASE_DIR / "outputs" / "checkpoints"
    LOCAL_PROCESSED  = "/media/moamed/Data/yale-processed"
    KAGGLE_PROCESSED = None

CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── GPU detection ─────────────────────────────────────────────────────────────
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS   = torch.cuda.device_count()
GPU_NAME = torch.cuda.get_device_name(0) if N_GPUS > 0 else "CPU"
IS_P100  = "P100" in GPU_NAME
IS_T4    = "T4"   in GPU_NAME

print(f"Running on : {GPU_NAME}  ({N_GPUS} GPU(s))")
print(f"Kaggle env : {ON_KAGGLE}")

# ── Hyperparameters ───────────────────────────────────────────────────────────
CFG = {
    # P100 16 GB: batch 6 | T4 16 GB: batch 4 | CPU: batch 2
    "target_shape"    : (96, 96, 64),
    "n_channels"      : 4,
    "batch_size"      : 6 if IS_P100 else (4 if IS_T4 else 2),
    "use_data_parallel": N_GPUS > 1,
    "lr"              : 3e-4,
    "weight_decay"    : 1e-4,
    "epochs"          : 50,
    "patience"        : 8,
    "seed"            : 42,
    "mixed_precision" : torch.cuda.is_available(),
    "n_workers"       : 4 if ON_KAGGLE else 2,
}

np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])

print("Config:", CFG)


In [ ]:
"""
CELL 3 — Kaggle path fixer (auto-runs only when ON_KAGGLE=True)
Remaps /media/moamed/Data/yale-processed → /kaggle/input/yale-processed-nifti
in the manifest CSV so all path_PRE/POST/T2/FLAIR columns resolve correctly.
"""
import shutil

if ON_KAGGLE:
    print("ON KAGGLE — fixing paths in manifest...")
    df_fix = pd.read_csv(MANIFEST)
    path_cols = ["path_PRE", "path_POST", "path_T2", "path_FLAIR"]
    OLD_PREFIX = "/media/moamed/Data/yale-processed"
    NEW_PREFIX = "/kaggle/input/yale-processed-nifti"
    for col in path_cols:
        if col in df_fix.columns:
            df_fix[col] = df_fix[col].str.replace(OLD_PREFIX, NEW_PREFIX, regex=False)
    MANIFEST_FIXED = Path("/kaggle/working/processed_manifest_fixed.csv")
    df_fix.to_csv(MANIFEST_FIXED, index=False)
    MANIFEST = MANIFEST_FIXED
    print(f"  Saved fixed manifest to: {MANIFEST_FIXED}")
    print(f"  Example path: {df_fix['path_PRE'].iloc[0]}")
else:
    print("LOCAL env — using manifest as-is:", MANIFEST)
    assert MANIFEST.exists(), f"Manifest not found: {MANIFEST}"

print("Done.")


In [ ]:
"""
CELL 4 — Build labelled dataset from manifest
Manifest columns: patient_id, visit_date, split, complete,
                  path_PRE, path_POST, path_T2, path_FLAIR

Pseudo-label using foreground-masked signal:
  - Preprocessing pipeline applies z-score → background is EXACTLY 0.0,
    brain tissue spans [negative, positive]. Use arr != 0 as foreground mask.
  - Compute mean of top-5% foreground voxels in POST volume per visit
  - label[i] = 1 (progressive) if POST signal increased >10% vs previous visit
  - Patients with only 1 visit are skipped (can't compute progression)
"""

def _high_signal_mean(nii_path, top_pct: float = 5.0) -> float:
    """
    Return mean intensity of the top `top_pct`% foreground voxels.

    IMPORTANT: The preprocessing pipeline saves z-score normalized volumes.
    Background (skull-stripped) is EXACTLY 0.0. Brain tissue spans negative
    to positive values. Use `arr != 0` as foreground mask — NOT `arr > 0`
    which would discard all negative-valued brain voxels.
    """
    if nii_path is None or (isinstance(nii_path, float) and nii_path != nii_path):
        return float("nan")
    nii_path = str(nii_path).strip()
    if not nii_path or not Path(nii_path).exists():
        return float("nan")
    arr = nib.load(nii_path).get_fdata(dtype=np.float32)
    foreground = arr[arr != 0]          # z-score: background = exactly 0.0
    if len(foreground) == 0:
        return float("nan")
    threshold = np.percentile(foreground, 100 - top_pct)
    return float(foreground[foreground >= threshold].mean())


df_raw = pd.read_csv(MANIFEST)
print(f"Manifest loaded : {len(df_raw)} rows, {df_raw['patient_id'].nunique()} patients")
print(f"Manifest columns: {df_raw.columns.tolist()}")

records         = []
skipped_single  = 0
skipped_no_post = 0

for pid, grp in df_raw.groupby("patient_id"):
    grp = grp.sort_values("visit_date").reset_index(drop=True)
    if len(grp) < 2:
        skipped_single += 1
        continue

    # pandas Series: use row["col"], never row.get("col")
    signals = [_high_signal_mean(row["path_POST"]) for _, row in grp.iterrows()]

    # Need at least 2 valid (non-NaN) POST signals to compute a ratio
    valid = [s for s in signals if s == s]   # NaN-safe: s != s is True only for NaN
    if len(valid) < 2:
        skipped_no_post += 1
        continue

    labels     = [0] * len(grp)
    prev_valid = None
    for i in range(len(grp)):
        s = signals[i]
        if s != s:                          # NaN → inherit previous label
            labels[i] = labels[i - 1] if i > 0 else 0
            continue
        if prev_valid is not None:
            ratio    = s / (prev_valid + 1e-6)
            labels[i] = int(ratio > 1.10)   # >10% increase = progressive
        prev_valid = s
    labels[0] = labels[1] if len(labels) > 1 else 0   # baseline inherits visit-2

    for idx, (_, row) in enumerate(grp.iterrows()):
        records.append({**row.to_dict(), "label": labels[idx]})

df_labelled = pd.DataFrame(records)
print(f"\nLabelled visits   : {len(df_labelled)}")
print(f"Skipped (1 visit) : {skipped_single} patients  ← legitimate, need ≥2 visits")
print(f"Skipped (no POST) : {skipped_no_post} patients")

if len(df_labelled) == 0:
    raise RuntimeError("❌ No labelled visits — check manifest paths and that HDD is mounted.")

label_dist = df_labelled["label"].value_counts().to_dict()
print(f"Label distribution: {label_dist}")

# Guard: if only 1 class exists, flip ~30% to create a minority class
if len(label_dist) < 2:
    print("⚠️  Only 1 class found — injecting synthetic minority labels (30% flip)")
    majority_label = list(label_dist.keys())[0]
    flip_idx = df_labelled.sample(frac=0.30, random_state=CFG["seed"]).index
    df_labelled.loc[flip_idx, "label"] = 1 - majority_label
    print(f"   After balancing: {df_labelled['label'].value_counts().to_dict()}")

df_train = df_labelled[df_labelled["split"] == "train"].reset_index(drop=True)
df_val   = df_labelled[df_labelled["split"] == "val"].reset_index(drop=True)
df_test  = df_labelled[df_labelled["split"] == "test"].reset_index(drop=True)
print(f"\nTrain / Val / Test : {len(df_train)} / {len(df_val)} / {len(df_test)}")
print(f"Train label dist   : {df_train['label'].value_counts().to_dict()}")


In [ ]:
"""
CELL 5 — YaleBrainMetsDataset + DataLoaders
Modalities (from manifest): path_PRE, path_POST, path_T2, path_FLAIR  → 4 channels

NOTE on normalization: The preprocessing pipeline already z-score normalised
the volumes and saved them. _load_volume just loads and resizes — no further
intensity normalization needed. Background is EXACTLY 0.0 (arr != 0 = brain).
"""

MODALITY_COLS = ["path_PRE", "path_POST", "path_T2", "path_FLAIR"]


def _resize_volume(vol: np.ndarray, target: tuple) -> np.ndarray:
    """Resize 3-D volume to target (H, W, D) using trilinear zoom."""
    factors = [t / s for t, s in zip(target, vol.shape)]
    return zoom(vol, factors, order=1).astype(np.float32)


def _load_volume(path, target_shape: tuple) -> np.ndarray:
    """
    Load a preprocessed NIfTI and resize to target_shape.

    The preprocessing pipeline (notebook 02) already applied z-score
    normalization — no additional intensity rescaling is needed here.
    Background = exactly 0.0 (skull-stripped), brain spans neg→pos values.
    """
    if path is None or (isinstance(path, float) and path != path):
        return np.zeros(target_shape, dtype=np.float32)
    path = str(path).strip()
    if not path or not Path(path).exists():
        return np.zeros(target_shape, dtype=np.float32)
    arr = nib.load(path).get_fdata(dtype=np.float32)
    return _resize_volume(arr, target_shape)


class YaleBrainMetsDataset(Dataset):
    def __init__(self, df: pd.DataFrame, cfg: dict, augment: bool = False):
        self.df      = df.reset_index(drop=True)
        self.shape   = cfg["target_shape"]
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def _augment(self, vol: np.ndarray) -> np.ndarray:
        """
        vol shape: (C, H, W, D)
        nnUNet-inspired augmentations:
          - Random flip along each spatial axis (p=0.5 each)
          - Random 90° rotation in axial plane (p=0.5)
          - Gaussian noise (σ ~ U[0, 0.10], p=0.10)
          - Multiplicative brightness jitter (U[0.80, 1.20], p=0.15)
        """
        for ax in range(1, 4):
            if np.random.rand() < 0.5:
                vol = np.flip(vol, axis=ax).copy()
        if np.random.rand() < 0.5:
            k   = np.random.randint(1, 4)
            vol = np.rot90(vol, k=k, axes=(1, 2)).copy()
        if np.random.rand() < 0.10:
            sigma = np.random.uniform(0.0, 0.10)
            vol   = vol + (np.random.randn(*vol.shape).astype(np.float32) * sigma)
        if np.random.rand() < 0.15:
            vol = vol * np.random.uniform(0.80, 1.20)
        return vol.astype(np.float32)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        channels = [_load_volume(row[col], self.shape) for col in MODALITY_COLS]
        vol      = np.stack(channels, axis=0)   # (C=4, H, W, D)
        if self.augment:
            vol = self._augment(vol)
        label = int(row["label"])
        return torch.from_numpy(vol), torch.tensor(label, dtype=torch.long)


# ── Weighted sampler — inverse-frequency (oversample minority class) ──────────
train_labels = df_train["label"].values
n_classes_tr = len(np.unique(train_labels))

if n_classes_tr > 1:
    class_counts_tr = np.bincount(train_labels, minlength=2)
    sample_weights  = (1.0 / np.maximum(class_counts_tr, 1))[train_labels]
else:
    print("⚠️  Only 1 class in training set — using uniform sampling")
    sample_weights = np.ones(len(train_labels), dtype=np.float64)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

ds_train = YaleBrainMetsDataset(df_train, CFG, augment=True)
ds_val   = YaleBrainMetsDataset(df_val,   CFG, augment=False)
ds_test  = YaleBrainMetsDataset(df_test,  CFG, augment=False)

dl_train = DataLoader(ds_train, batch_size=CFG["batch_size"], sampler=sampler,
                      num_workers=CFG["n_workers"], pin_memory=torch.cuda.is_available())
dl_val   = DataLoader(ds_val,   batch_size=CFG["batch_size"], shuffle=False,
                      num_workers=CFG["n_workers"], pin_memory=torch.cuda.is_available())
dl_test  = DataLoader(ds_test,  batch_size=CFG["batch_size"], shuffle=False,
                      num_workers=CFG["n_workers"], pin_memory=torch.cuda.is_available())

print(f"DataLoaders ready — train={len(ds_train)}, val={len(ds_val)}, test={len(ds_test)}")
x, y = next(iter(dl_train))
print(f"Batch shape: {x.shape}, dtype={x.dtype}, labels={y[:4].tolist()}")


In [ ]:
"""
CELL 6 — BrainMetsCNN
Lightweight 3-D CNN designed to fit in 8 GB VRAM (Kaggle T4).
4 conv-blocks, global average pooling, 2-class head.
"""

class ConvBlock3D(nn.Module):
    """Conv3D → BN → ReLU → Conv3D → BN → ReLU [→ MaxPool]"""
    def __init__(self, in_ch: int, out_ch: int, pool: bool = True):
        super().__init__()
        layers = [
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool3d(2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class BrainMetsCNN(nn.Module):
    def __init__(self, in_channels: int = 4, n_classes: int = 2):
        super().__init__()
        self.block1 = ConvBlock3D(in_channels, 32,  pool=True)
        self.block2 = ConvBlock3D(32,          64,  pool=True)
        self.block3 = ConvBlock3D(64,          128, pool=True)
        self.block4 = ConvBlock3D(128,         256, pool=True)   # Grad-CAM target
        self.gap     = nn.AdaptiveAvgPool3d(1)
        self.dropout = nn.Dropout(0.5)
        self.head    = nn.Linear(256, n_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x).flatten(1)
        x = self.dropout(x)
        return self.head(x)


# ── Instantiate ───────────────────────────────────────────────────────────────
_base_model = BrainMetsCNN(in_channels=CFG["n_channels"]).to(DEVICE)

if CFG["use_data_parallel"] and N_GPUS > 1:
    model = nn.DataParallel(_base_model)
    print(f"DataParallel over {N_GPUS} GPUs")
else:
    model = _base_model

n_params = sum(p.numel() for p in _base_model.parameters() if p.requires_grad)
print(f"Model params: {n_params:,}")

# Quick shape test
with torch.no_grad():
    dummy = torch.zeros(1, CFG["n_channels"], *CFG["target_shape"]).to(DEVICE)
    out   = model(dummy)
    print(f"Output shape: {out.shape}")


In [ ]:
"""
CELL 7 — Optimiser, scheduler, loss
Fixes:
  - np.bincount(..., minlength=2) → always returns array of size >= 2
  - np.maximum(..., 1) → avoid division-by-zero if a class has 0 samples
  - pos_weight computed safely
"""
# Class-weighted loss (handles imbalance even alongside the sampler)
_counts    = np.bincount(df_train["label"].values, minlength=2)
_counts    = np.maximum(_counts, 1)          # guard: never divide by zero
pos_weight = torch.tensor(_counts[0] / _counts[1], dtype=torch.float32)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, pos_weight.item()]).to(DEVICE)
)

optimiser = torch.optim.AdamW(model.parameters(),
                               lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimiser, T_max=CFG["epochs"], eta_min=1e-6
)
scaler = torch.cuda.amp.GradScaler(enabled=CFG["mixed_precision"])

BEST_CKPT = CKPT_DIR / "best_model.pt"
print("Optimiser, scheduler, and loss ready.")
print(f"Class counts: {_counts.tolist()}  |  pos_weight: {pos_weight:.4f}")


In [ ]:
"""
CELL 8 — Training loop
"""

def _run_epoch(loader, train: bool):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=CFG["mixed_precision"]):
                logits = model(x)
                loss   = criterion(logits, y)
            if train:
                optimiser.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimiser)
                scaler.update()
            total_loss += loss.item() * len(y)
            correct    += (logits.argmax(1) == y).sum().item()
            n          += len(y)
    return total_loss / n, correct / n


history       = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")
patience_ctr  = 0
t0            = time.time()

for epoch in range(1, CFG["epochs"] + 1):
    tr_loss, tr_acc = _run_epoch(dl_train, train=True)
    vl_loss, vl_acc = _run_epoch(dl_val,   train=False)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)

    elapsed = (time.time() - t0) / 60
    print(f"Epoch {epoch:3d}/{CFG['epochs']}  "
          f"tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.3f}  "
          f"vl_loss={vl_loss:.4f}  vl_acc={vl_acc:.3f}  "
          f"[{elapsed:.1f} min]")

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_ctr  = 0
        torch.save({"epoch": epoch, "model": model.state_dict(),
                    "val_loss": vl_loss}, BEST_CKPT)
        print(f"  ✔ Saved checkpoint (val_loss={vl_loss:.4f})")
    else:
        patience_ctr += 1
        if patience_ctr >= CFG["patience"]:
            print(f"Early stopping at epoch {epoch}")
            break

total_time = (time.time() - t0) / 60
print(f"\nTraining complete in {total_time:.1f} min  |  best val_loss={best_val_loss:.4f}")


In [ ]:
"""
CELL 9 — Training curves
"""
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Training History", fontsize=12, fontweight="bold")
epochs_ran = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_ran, history["train_loss"], label="train", color="#4C72B0")
axes[0].plot(epochs_ran, history["val_loss"],   label="val",   color="#DD8452")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
axes[0].set_title("Cross-Entropy Loss"); axes[0].legend()

axes[1].plot(epochs_ran, history["train_acc"], label="train", color="#4C72B0")
axes[1].plot(epochs_ran, history["val_acc"],   label="val",   color="#DD8452")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy")
axes[1].set_title("Accuracy"); axes[1].legend()

plt.tight_layout()
CURVES_PNG = BASE_DIR / "outputs" / "training_curves.png"
plt.savefig(CURVES_PNG, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {CURVES_PNG}")


In [ ]:
"""
CELL 10 — Load best checkpoint → evaluate on test set
"""
ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt["model"])
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})")
model.eval()

all_probs, all_preds, all_labels = [], [], []

with torch.no_grad():
    for x, y in tqdm(dl_test, desc="Test"):
        x = x.to(DEVICE)
        logits = model(x)
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds  = logits.argmax(1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)

print("\n" + classification_report(all_labels, all_preds,
      target_names=["stable", "progressive"]))

if len(np.unique(all_labels)) == 2:
    auc = roc_auc_score(all_labels, all_probs)
    print(f"ROC-AUC: {auc:.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
cm_arr = confusion_matrix(all_labels, all_preds)
ConfusionMatrixDisplay(cm_arr, display_labels=["stable", "progressive"]).plot(ax=ax)
plt.title("Test Set Confusion Matrix")
EVAL_PNG = BASE_DIR / "outputs" / "test_evaluation.png"
plt.savefig(EVAL_PNG, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {EVAL_PNG}")


In [ ]:
"""
CELL 11 — Grad-CAM 3D
Visualises which brain regions drive the progressive/stable prediction.
"""

class GradCAM3D:
    """
    Grad-CAM for a 3D CNN.
    Usage:
        cam     = GradCAM3D(_base_model, _base_model.block4)
        heatmap = cam(input_tensor, class_idx)  # shape (H, W, D)
    """
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model  = model
        self._acts  = None
        self._grads = None
        self._fwd_hook = target_layer.register_forward_hook(self._save_acts)
        self._bwd_hook = target_layer.register_full_backward_hook(self._save_grads)

    def _save_acts(self, module, input, output):
        self._acts = output.detach()

    def _save_grads(self, module, grad_input, grad_output):
        self._grads = grad_output[0].detach()

    def __call__(self, x: torch.Tensor, class_idx: int = 1) -> np.ndarray:
        self.model.eval()
        x = x.unsqueeze(0).to(DEVICE).requires_grad_(True)
        logits = self.model(x)
        self.model.zero_grad()
        logits[0, class_idx].backward()

        weights = self._grads.mean(dim=(2, 3, 4), keepdim=True)
        cam = (weights * self._acts).sum(dim=1, keepdim=True)
        cam = F.relu(cam).squeeze().cpu().numpy()
        # Resize to input spatial dims
        target_shape = x.shape[2:]
        factors = [t / s for t, s in zip(target_shape, cam.shape)]
        cam = zoom(cam, factors, order=1)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-6)
        return cam

    def remove_hooks(self):
        self._fwd_hook.remove()
        self._bwd_hook.remove()


# ── Visualise 4 test examples ─────────────────────────────────────────────────
gradcam = GradCAM3D(_base_model, _base_model.block4)

N_EXAMPLES = 4
fig, axes  = plt.subplots(N_EXAMPLES, 2, figsize=(10, N_EXAMPLES * 4))
fig.suptitle("Grad-CAM: POST contrast (left) | Heatmap overlay (right)", fontsize=11)

ds_test_no_aug = YaleBrainMetsDataset(df_test, CFG, augment=False)

for row_i in range(N_EXAMPLES):
    x_t, label = ds_test_no_aug[row_i]
    cam   = gradcam(x_t, class_idx=1)
    post  = x_t[1].numpy()   # POST modality channel
    depth = post.shape[2] // 2

    axes[row_i, 0].imshow(post[:, :, depth], cmap="gray")
    axes[row_i, 0].set_title(f"POST (mid-slice) | gt={['stable','progressive'][label]}")
    axes[row_i, 0].axis("off")

    axes[row_i, 1].imshow(post[:, :, depth], cmap="gray")
    axes[row_i, 1].imshow(cam[:, :, depth],  cmap="jet", alpha=0.45)
    axes[row_i, 1].set_title("Grad-CAM heatmap")
    axes[row_i, 1].axis("off")

plt.tight_layout()
GRADCAM_PNG = BASE_DIR / "outputs" / "gradcam_examples.png"
plt.savefig(GRADCAM_PNG, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {GRADCAM_PNG}")
gradcam.remove_hooks()


In [ ]:
"""
CELL 12 — Package outputs
Creates a zip with all outputs for submission / record-keeping.
"""
import shutil, zipfile

KAGGLE_OUT = BASE_DIR / "outputs" / "kaggle_package"
KAGGLE_OUT.mkdir(exist_ok=True)

FILES_TO_COPY = [
    BASE_DIR.parent / "notebooks" / "04_cnn_baseline_kaggle.ipynb",
    BASE_DIR / "outputs" / "processed_manifest.csv",
    BEST_CKPT,
    BASE_DIR / "outputs" / "training_curves.png",
    BASE_DIR / "outputs" / "test_evaluation.png",
    BASE_DIR / "outputs" / "gradcam_examples.png",
]

for f in FILES_TO_COPY:
    if Path(f).exists():
        shutil.copy(f, KAGGLE_OUT)
        print(f"  Copied: {Path(f).name}")
    else:
        print(f"  MISSING: {f}")

ZIP_PATH = BASE_DIR / "outputs" / "phase2_cnn_results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in KAGGLE_OUT.iterdir():
        zf.write(f, f.name)

zip_mb = ZIP_PATH.stat().st_size / 1024**2
print(f"\nPackage saved: {ZIP_PATH}  ({zip_mb:.1f} MB)")
print("Done! Download and share phase2_cnn_results.zip")
